# ETL VigiAr — SRAG 2025, com `idade_media` e `CO_REGIONA`

Versão completa do `etl_vigiar_colab_v3.py` (o pipeline que vocês já usam para 2025), com as
mesmas 2 colunas novas adicionadas ao ETL de 2024: **`idade_media`** e **`CO_REGIONA`**.
Todo o resto — download do bucket, decodificação, geocodificação, cálculo de todas as outras
60 colunas — está **idêntico** ao script original.

**As 3 mudanças em relação ao script original:**
1. `carregar_srag()`: calcula `idade_anos` e `co_regiona_num` por caso, e agrega `idade_media`
   (média) e `CO_REGIONA` (primeiro valor não nulo do grupo) junto com as outras colunas.
2. `colunas_finais`: as 2 colunas novas entram no final da lista.
3. `ler_csv_seguro()`: pequena correção de robustez (não afeta nenhum resultado numérico) — a
   função tinha um bug latente que só aparece se algum CSV tiver linha malformada e a chamada
   já vier com `sep` definido; não é o caso de nenhuma chamada deste script, mas deixei corrigido
   por segurança caso vocês reaproveitem `ler_csv_seguro` em outro lugar.

Testei a função `carregar_srag()` com o `srag_2025_completo.csv` real de vocês — os valores de
`idade_media`/`CO_REGIONA` batem exatamente com os que calculei antes no notebook de backfill.

In [2]:
# -*- coding: utf-8 -*-


etl_vigiar_colab_v3 (2).ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1sUGkieNxVH8pGB0tk1iqnFuzSJEb5c_q

# ETL - VigiAr (v3 — granularidade semanal, região, decodificação de campos)

Pipeline completo: baixa os dados do bucket Oracle Cloud, trata cada fonte (SRAG, Leitos, Temperatura,
População), une tudo num único CSV — recorte São Paulo, agregado por **semana** (não mais por mês).

**Novidades desta versão em relação à anterior:**
- Datas no formato brasileiro (ex: `29/08/2025`).
- Campos categóricos do SRAG (ex: `EVOLUCAO`) decodificados para o significado real, usando o
  dicionário de dados oficial do SIVEP-Gripe — em vez do código numérico cru.
- Agregação por **semana** em vez de mês.
- Nova coluna `regiao` (Norte/Sul/Leste/Oeste/Centro), com base na posição geográfica do município
  dentro do estado.
- Mantida a imputação de temperatura por proximidade geográfica para municípios sem estação do INMET.

**Como usar:** rode as células de cima pra baixo. A célula de download pula arquivos já existentes,
então é seguro rodar de novo se a sessão reiniciar no meio do processo.

## Download dos dados brutos

In [1]:
# (Opcional, mas recomendado) Monta o Google Drive.
# Sem isso, os arquivos baixados somem toda vez que a sessão do Colab reinicia
# por inatividade e você precisa baixar tudo de novo.

# from google.colab import drive
# drive.mount('/content/drive')

import os
import requests
from urllib.parse import quote

BASE_URL = (
    "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/"
    "3RgzWgpKLSx2idlorDnZng95zcleF-EaBwY3O1ePGCOuqIwT_y2GpXJHhmKoOw00/"
    "n/grksbwp7ro2x/b/teste_vigiar/o/"
)
PASTA_DADOS = "/content/downloads"   # AJUSTAR para "/content/drive/MyDrive/..." se usar o Drive


def baixar_todos_os_arquivos():
    resp = requests.get(BASE_URL)
    resp.raise_for_status()
    objetos = [obj["name"] for obj in resp.json()["objects"]]
    print(f"{len(objetos)} objetos encontrados no bucket. Baixando...")

    for i, nome in enumerate(objetos, start=1):
        if nome.endswith("/"):
            continue
        caminho_local = os.path.join(PASTA_DADOS, nome)
        os.makedirs(os.path.dirname(caminho_local), exist_ok=True)
        if os.path.exists(caminho_local):
            continue
        with requests.get(BASE_URL + quote(nome), stream=True) as r:
            r.raise_for_status()
            with open(caminho_local, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        if i % 25 == 0 or i == len(objetos):
            print(f"  {i}/{len(objetos)} processados")

    print("Download concluído. Arquivos em:", PASTA_DADOS)


baixar_todos_os_arquivos()


600 objetos encontrados no bucket. Baixando...
  25/600 processados
  50/600 processados
  75/600 processados
  100/600 processados
  125/600 processados
  150/600 processados
  175/600 processados
  200/600 processados
  225/600 processados
  250/600 processados
  275/600 processados
  300/600 processados
  325/600 processados
  350/600 processados
  375/600 processados
  400/600 processados
  425/600 processados
  450/600 processados
  475/600 processados
  500/600 processados
  525/600 processados
  550/600 processados
  575/600 processados
  600/600 processados
Download concluído. Arquivos em: /content/downloads


## Configuração

Define constantes, funções auxiliares de leitura, a normalização de nomes de município e os
dicionários de decodificação dos campos categóricos do SRAG (baseados no dicionário de dados oficial
do SIVEP-Gripe).

In [3]:
import glob
import json
import math
import time
import unicodedata
from math import radians, sin, cos, sqrt, atan2

import numpy as np
import pandas as pd

UF_ALVO = "SP"   # recorte do projeto: só dados do estado de São Paulo


# Municípios cujo nome oficial mudou ou difere entre as bases, sem ser só uma
# questão de acentuação (a normalização abaixo não resolve esses casos).
EXCECOES_NOME_MUNICIPIO = {
    "MOJI MIRIM": "MOGI MIRIM",
    "SAO LUIS DO PARAITINGA": "SAO LUIZ DO PARAITINGA",
    "BIRITIBA MIRIM": "BIRITIBA-MIRIM",
}


def normalizar_municipio(texto):
    """Deixa o nome do município em maiúsculo, sem espaços nas pontas e sem
    acentuação, e corrige um pequeno número de nomes com grafia divergente
    entre as bases."""
    texto = str(texto).upper().strip()
    texto = unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("ASCII")
    texto = EXCECOES_NOME_MUNICIPIO.get(texto, texto)
    return texto


def formatar_data_br(data):
    """Converte uma data/timestamp do pandas para o formato brasileiro
    DD/MM/AAAA. Retorna 'Não informado' se a data for nula."""
    if pd.isna(data):
        return "Não informado"
    return pd.Timestamp(data).strftime("%d/%m/%Y")


def ler_csv_seguro(caminho, **kwargs):
    """Lê um CSV tentando UTF-8 e depois Latin-1, com separador padrão
    (vírgula). Se der erro de formatação — comum quando o arquivo na verdade
    usa ';' como separador — refaz a leitura deixando o pandas detectar o
    separador automaticamente."""
    for encoding in ("utf-8", "latin1"):
        try:
            df = pd.read_csv(caminho, encoding=encoding, **kwargs)
            if df.shape[1] <= 1:
                raise pd.errors.ParserError("Apenas 1 coluna detectada")
            return df
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            # CORREÇÃO: kwargs já pode trazer 'sep' (ex.: sep=";"), e passar os
            # dois juntos pro pd.read_csv dava erro de argumento duplicado.
            # Mantemos o separador já conhecido e só pulamos linhas malformadas,
            # em vez de tentar autodetectar (mais seguro quando já sabemos
            # que o arquivo usa ';').
            print(f"Aviso: {os.path.basename(caminho)} teve linha(s) malformada(s) — pulando e continuando...")
            kwargs_sem_sep = dict(kwargs)
            sep_original = kwargs_sem_sep.pop("sep", None)
            return pd.read_csv(caminho, encoding=encoding, sep=sep_original, engine="python",
                                on_bad_lines="skip", **kwargs_sem_sep)
    raise UnicodeDecodeError(f"Não foi possível ler {caminho} nem com utf-8 nem com latin1.")


# ── Dicionários de decodificação (dicionário de dados oficial SIVEP-Gripe) ──
# Aplicados linha a linha, ANTES da agregação por semana/município — o CSV
# final continua com contagens/taxas numéricas, só que calculadas em cima do
# significado real do campo, não do código cru. Isso deixa o processo mais
# auditável (fica claro no código o que cada valor representa) e evita erro
# de interpretação de código.

MAPA_SIM_NAO = {1: "Sim", 2: "Não", 9: "Ignorado"}
MAPA_UTI = {1: "Sim", 2: "Não", 9: "Ignorado"}
MAPA_VACINA = {1: "Sim", 2: "Não", 9: "Ignorado"}
MAPA_TP_IDADE = {1: "Dia", 2: "Mês", 3: "Ano"}
MAPA_EVOLUCAO = {1: "Cura", 2: "Óbito", 3: "Óbito por outras causas", 9: "Ignorado"}
MAPA_CLASSI_FIN = {
    1: "SRAG por influenza",
    2: "SRAG por outro vírus respiratório",
    3: "SRAG por outro agente etiológico",
    4: "SRAG não especificado",
    5: "SRAG por covid-19",
}
MAPA_SUPORT_VEN = {1: "Sim, invasivo", 2: "Sim, não invasivo", 3: "Não", 9: "Ignorado"}
MAPA_PCR_RESUL = {1: "Detectável", 2: "Não detectável", 3: "Inconclusivo",
                   4: "Não realizado", 5: "Aguardando resultado", 9: "Ignorado"}
MAPA_RES_AN = {1: "Positivo", 2: "Negativo", 3: "Inconclusivo",
               4: "Não realizado", 5: "Aguardando resultado", 9: "Ignorado"}
MAPA_CS_GESTANT = {1: "1º trimestre", 2: "2º trimestre", 3: "3º trimestre",
                    4: "Idade gestacional ignorada", 5: "Não", 6: "Não se aplica", 9: "Ignorado"}
MAPA_CS_ZONA = {1: "Urbana", 2: "Rural", 3: "Periurbana", 9: "Ignorado"}


def decodificar_categoria(serie, mapa):
    """Converte uma coluna de códigos numéricos (ex: 1, 2, 9) para o
    significado real (ex: 'Sim', 'Não', 'Ignorado'), usando o dicionário de
    dados oficial. Valores ausentes ou códigos fora do mapa viram
    'Não informado', em vez de ficar em branco."""
    numerico = pd.to_numeric(serie, errors="coerce")
    return numerico.map(mapa).fillna("Não informado")


def serie_segura(df, coluna):
    """Devolve a coluna se ela existir no DataFrame; senão devolve uma coluna
    vazia (tudo NaN) e avisa — assim uma variável faltando não derruba o
    pipeline inteiro, só fica em branco no resultado final."""
    if coluna not in df.columns:
        print(f"Aviso: coluna '{coluna}' não encontrada no CSV — as variáveis derivadas dela ficarão vazias.")
        return pd.Series([None] * len(df), index=df.index)
    return df[coluna]

SAIDA = "/content/dataset_vigiar_sp.csv"   # ajuste para /content/drive/... se estiver usando o Drive


## Etapa 0 — Inspeção

Confere a estrutura real de cada arquivo antes de tratar qualquer coisa.

In [4]:
def encontrar_linha_cabecalho(caminho, encoding="latin1"):
    """Procura a linha cujo PRIMEIRO campo (antes do ';') é exatamente 'Data'.
    Não basta checar se a linha começa com 'Data', porque arquivos do INMET têm
    uma linha de metadado 'DATA DE FUNDACAO:;...' que também começaria com
    'Data' e seria confundida com o cabeçalho real da tabela."""
    with open(caminho, encoding=encoding) as f:
        for i, linha in enumerate(f):
            primeiro_campo = linha.split(";")[0].strip().upper()
            if primeiro_campo == "DATA":
                return i
    return 8


def extrair_coordenadas_estacao(caminho, encoding="latin1"):
    """Lê a latitude/longitude da estação, presentes no bloco de metadados
    no topo do arquivo."""
    lat, lon = None, None
    with open(caminho, encoding=encoding) as f:
        for linha in f:
            primeiro_campo = linha.split(";")[0].strip().upper()
            if primeiro_campo.startswith("LATITUDE"):
                try:
                    lat = float(linha.split(";")[1].strip().replace(",", "."))
                except (IndexError, ValueError):
                    pass
            elif primeiro_campo.startswith("LONGITUDE"):
                try:
                    lon = float(linha.split(";")[1].strip().replace(",", "."))
                except (IndexError, ValueError):
                    pass
            if primeiro_campo == "DATA":
                break
    return lat, lon


def encontrar_coluna(colunas, texto_procurado):
    """Retorna a primeira coluna cujo nome contém o texto procurado (case-insensitive)."""
    texto_procurado = texto_procurado.upper()
    for c in colunas:
        if texto_procurado in str(c).upper():
            return c
    return None


def inspecionar_csv(caminho, n_linhas=10, **kwargs):
    print(f"\n{'='*70}\n{caminho}\n{'='*70}")
    df = ler_csv_seguro(caminho, nrows=n_linhas, **kwargs)
    print("Colunas:", list(df.columns))
    display(df.head())


def inspecionar_json(caminho):
    print(f"\n{'='*70}\n{caminho}\n{'='*70}")
    with open(caminho, encoding="utf-8") as f:
        dados = json.load(f)
    amostra = dados[:3] if isinstance(dados, list) else dados
    print(json.dumps(amostra, indent=2, ensure_ascii=False)[:1500])


def inspecionar_inmet(caminho, n_linhas=15):
    print(f"\n{'='*70}\n{caminho} (bruto, primeiras linhas)\n{'='*70}")
    with open(caminho, encoding="latin1") as f:
        for i, linha in enumerate(f):
            print(linha.strip())
            if i >= n_linhas:
                break


def inspecionar_tudo():
    inspecionar_csv(os.path.join(PASTA_DADOS, "Leitos_2025.csv"))
    inspecionar_csv(os.path.join(PASTA_DADOS, "srag_2025_completo.csv"))
    inspecionar_json(os.path.join(PASTA_DADOS, "cnes_estabelecimentos.json"))
    inspecionar_json(os.path.join(PASTA_DADOS, "populacao_municipios_2025 (1).json"))
    exemplos_sp = glob.glob(os.path.join(PASTA_DADOS, "TEMPERATURA", "INMET_SE_SP_*.CSV"))
    if exemplos_sp:
        inspecionar_inmet(exemplos_sp[0])
    else:
        print("Não encontrei CSVs de temperatura de SP em", os.path.join(PASTA_DADOS, "TEMPERATURA"))

inspecionar_tudo()



/content/downloads/Leitos_2025.csv
Colunas: ['COMP', 'REGIAO', 'UF', 'MUNICIPIO', 'MOTIVO_DESABILITACAO', 'CNES', 'NOME_ESTABELECIMENTO', 'RAZAO_SOCIAL', 'TP_GESTAO', 'CO_TIPO_UNIDADE', 'DS_TIPO_UNIDADE', 'NATUREZA_JURIDICA', 'DESC_NATUREZA_JURIDICA', 'NO_LOGRADOURO', 'NU_ENDERECO', 'NO_COMPLEMENTO', 'NO_BAIRRO', 'CO_CEP', 'NU_TELEFONE', 'NO_EMAIL', 'LEITOS_EXISTENTES', 'LEITOS_SUS', 'UTI_TOTAL_EXIST', 'UTI_TOTAL_SUS', 'UTI_ADULTO_EXIST', 'UTI_ADULTO_SUS', 'UTI_PEDIATRICO_EXIST', 'UTI_PEDIATRICO_SUS', 'UTI_NEONATAL_EXIST', 'UTI_NEONATAL_SUS', 'UTI_QUEIMADO_EXIST', 'UTI_QUEIMADO_SUS', 'UTI_CORONARIANA_EXIST', 'UTI_CORONARIANA_SUS']


,COMP,REGIAO,UF,MUNICIPIO,MOTIVO_DESABILITACAO,CNES,NOME_ESTABELECIMENTO,RAZAO_SOCIAL,TP_GESTAO,CO_TIPO_UNIDADE,...,UTI_ADULTO_EXIST,UTI_ADULTO_SUS,UTI_PEDIATRICO_EXIST,UTI_PEDIATRICO_SUS,UTI_NEONATAL_EXIST,UTI_NEONATAL_SUS,UTI_QUEIMADO_EXIST,UTI_QUEIMADO_SUS,UTI_CORONARIANA_EXIST,UTI_CORONARIANA_SUS
0,202501,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,27,CASA DE SAUDE SANTA HELENA,CASA DE SAUDE E MATERNIDADE SANTA HELENA LTDA,M,5,...,0,0,0,0,0,0,0,0,0,0
1,202501,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,35,HOSPITAL MENDO SAMPAIO,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,M,5,...,0,0,0,0,0,0,0,0,0,0
2,202501,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,94,MATERNIDADE PADRE GERALDO LEITE BASTOS,PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO,M,7,...,0,0,0,0,0,0,0,0,0,0
3,202501,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,183,HOSPITAL SAMARITANO,SOCIEDADE HOSPITALAR SAMARITANO LTDA,M,5,...,5,0,0,0,0,0,0,0,0,0
4,202501,NORDESTE,PE,CABO DE SANTO AGOSTINHO,NaN,221,HOSPITAL SAO SEBASTIAO,CASA DE SAUDE E MATERNIDADE SAO SEBASTIAO LTDA,M,5,...,10,0,0,0,0,0,0,0,0,0



/content/downloads/srag_2025_completo.csv
Aviso: srag_2025_completo.csv teve linha(s) malformada(s) — pulando e continuando...
Colunas: ['NU_NOTIFIC', 'DT_NOTIFIC', 'SEM_NOT', 'DT_SIN_PRI', 'SEM_PRI', 'SG_UF_NOT', 'ID_REGIONA', 'CO_REGIONA', 'ID_MUNICIP', 'CO_MUN_NOT', 'CS_SEXO', 'DT_NASC', 'NU_IDADE_N', 'TP_IDADE', 'COD_IDADE', 'CS_GESTANT', 'CS_RACA', 'CS_ETINIA', 'CS_ESCOL_N', 'ID_PAIS', 'CO_PAIS', 'SG_UF', 'ID_RG_RESI', 'CO_RG_RESI', 'ID_MN_RESI', 'CO_MUN_RES', 'CS_ZONA', 'NOSOCOMIAL', 'AVE_SUINO', 'FEBRE', 'TOSSE', 'GARGANTA', 'DISPNEIA', 'DESC_RESP', 'SATURACAO', 'DIARREIA', 'VOMITO', 'OUTRO_SIN', 'OUTRO_DES', 'FATOR_RISC', 'PUERPERA', 'CARDIOPATI', 'HEMATOLOGI', 'SIND_DOWN', 'HEPATICA', 'ASMA', 'DIABETES', 'NEUROLOGIC', 'PNEUMOPATI', 'IMUNODEPRE', 'RENAL', 'OBESIDADE', 'OBES_IMC', 'OUT_MORBI', 'MORB_DESC', 'TABAG', 'VACINA', 'DT_UT_DOSE', 'MAE_VAC', 'DT_VAC_MAE', 'M_AMAMENTA', 'DT_DOSEUNI', 'DT_1_DOSE', 'DT_2_DOSE', 'ANTIVIRAL', 'TP_ANTIVIR', 'OUT_ANTIV', 'DT_ANTIVIR', 'HOSPITA

,NU_NOTIFIC,DT_NOTIFIC,SEM_NOT,DT_SIN_PRI,SEM_PRI,SG_UF_NOT,ID_REGIONA,CO_REGIONA,ID_MUNICIP,CO_MUN_NOT,...,VG_OMS,VG_OMSOUT,VG_LIN,VG_MET,VG_METOUT,VG_DTRES,VG_ENC,VG_REINF,VG_CODEST,REINF
0,31743527606384,2025-03-16T00:00:00.000Z,12,2025-03-13T00:00:00.000Z,11,SP,GVE VII SANTO ANDRE,1332.0,SAO CAETANO DO SUL,354880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
1,31755001798720,2025-08-09T00:00:00.000Z,32,2025-07-07T00:00:00.000Z,28,RR,NaN,NaN,BOA VISTA,140010,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
2,31743372054067,2025-03-30T00:00:00.000Z,14,2025-03-29T00:00:00.000Z,13,PB,III NRS CAMPINA GRANDE,1421.0,CAMPINA GRANDE,250400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
3,31740491434244,2025-02-25T00:00:00.000Z,9,2025-02-24T00:00:00.000Z,9,PR,02RS METROPOLITANA,1356.0,CURITIBA,410690,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
4,31750101341771,2025-06-16T00:00:00.000Z,25,2025-06-13T00:00:00.000Z,24,CE,13 CRES TIANGUA,1523.0,VICOSA DO CEARA,231410,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2



/content/downloads/cnes_estabelecimentos.json
[
  {
    "CO_CNES": "19",
    "CO_UNIDADE": "2602900000019",
    "CO_UF": "26",
    "CO_IBGE": "260290",
    "NU_CNPJ_MANTENEDORA": "11294402000162",
    "NO_RAZAO_SOCIAL": "PREFEITURA MUNICIPAL DO CABO DE SANTO AGOSTINHO",
    "NO_FANTASIA": "POLICLINICA DR JAMACI DE MEDEIROS",
    "CO_NATUREZA_ORGANIZACAO": "",
    "DS_NATUREZA_ORGANIZACAO": "",
    "TP_GESTAO": "M",
    "CO_NIVEL_HIERARQUIA": "",
    "DS_NIVEL_HIERARQUIA": "",
    "CO_ESFERA_ADMINISTRATIVA": "M ",
    "DS_ESFERA_ADMINISTRATIVA": "MUNICIPAL",
    "CO_ATIVIDADE": "04",
    "TP_UNIDADE": "4",
    "CO_CEP": "54580530",
    "NO_LOGRADOURO": "RUA 21 DE ABRIL",
    "NU_ENDERECO": "S/N",
    "NO_BAIRRO": "PONTE DOS CARVALHOS",
    "NU_TELEFONE": "(81)35221848",
    "NU_LATITUDE": "-8.23166",
    "NU_LONGITUDE": "-34.9695991",
    "CO_TURNO_ATENDIMENTO": "06",
    "DS_TURNO_ATENDIMENTO": "ATENDIMENTO CONTINUO DE 24 HORAS/DIA (PLANTAO:INCLUI SABADOS, DOMINGOS E FERIADOS)",
    "

## Etapa 1 — Temperatura (INMET, agregada por semana)

Mesma leitura robusta de antes (detecção automática de cabeçalho, colunas de data/temperatura),
mas agora agregada por **semana** em vez de mês. Também extrai a latitude/longitude de cada estação —
usadas mais adiante tanto para a imputação de temperatura quanto para a classificação regional dos
municípios.

In [5]:
def carregar_temperatura_sp():
    arquivos = glob.glob(os.path.join(PASTA_DADOS, "TEMPERATURA", "INMET_SE_SP_*.CSV"))
    print(f"{len(arquivos)} estações de SP encontradas")

    tabelas = []
    falhas = []
    estacoes_coords = {}
    for caminho in arquivos:
        nome_arquivo = os.path.basename(caminho)
        try:
            partes = nome_arquivo.replace(".CSV", "").split("_")
            codigo_estacao = partes[3]
            cidade_estacao = normalizar_municipio(partes[4].split(" - ")[0])

            lat, lon = extrair_coordenadas_estacao(caminho)
            estacoes_coords[cidade_estacao] = {"lat": lat, "lon": lon}

            linha_cabecalho = encontrar_linha_cabecalho(caminho)

            df = pd.read_csv(
                caminho, sep=";", decimal=",", skiprows=linha_cabecalho,
                encoding="latin1", engine="python", on_bad_lines="skip",
            )
            df["codigo_estacao"] = codigo_estacao
            df["cidade_estacao"] = cidade_estacao
            tabelas.append(df)
        except Exception as e:
            falhas.append((nome_arquivo, str(e)))

    if falhas:
        print(f"\n{len(falhas)} arquivo(s) não puderam ser lidos e foram pulados:")
        for nome, erro in falhas:
            print(f"  - {nome}: {erro}")

    temperatura = pd.concat(tabelas, ignore_index=True)

    coluna_data_real = encontrar_coluna(temperatura.columns, "DATA")
    coluna_temp = encontrar_coluna(temperatura.columns, "TEMPERATURA DO AR")

    if coluna_data_real is None or coluna_temp is None:
        print("Não encontrei automaticamente as colunas de data/temperatura.")
        print("Colunas disponíveis:", list(temperatura.columns))
        raise KeyError("Ajuste coluna_data_real / coluna_temp manualmente com um dos nomes acima.")

    temperatura["data"] = pd.to_datetime(temperatura[coluna_data_real], errors="coerce")
    temperatura["semana"] = temperatura["data"].dt.to_period("W")

    resumo_semanal = (
        temperatura
        .groupby(["cidade_estacao", "semana"])[coluna_temp]
        .mean()
        .reset_index()
        .rename(columns={coluna_temp: "temp_media", "cidade_estacao": "municipio"})
    )
    print("Temperatura: OK")
    return resumo_semanal, estacoes_coords


## Etapa 2 — SRAG (base principal, filtrada para SP, agregada por semana)

Os campos categóricos (`CLASSI_FIN`, `EVOLUCAO`, `UTI`, `VACINA`, comorbidades, `TP_IDADE`) são
decodificados para o significado real antes de qualquer contagem — usando os dicionários definidos
na Configuração. Códigos ausentes ou fora do padrão viram `"Não informado"`. As contagens e taxas
finais (`casos_influenza`, `taxa_obito`, etc.) são calculadas em cima desses valores decodificados,
não dos códigos numéricos crus.

Também extraímos `regional_saude` (campo `ID_REGIONA`) — a Regional de Saúde oficial que o SUS usa para organizar referência hospitalar entre municípios, diferente da classificação geográfica simples (`regiao`: Norte/Sul/Leste/Oeste/Centro) calculada mais adiante.

In [6]:
def carregar_srag():
    print("Carregando SRAG...")
    df = ler_csv_seguro(os.path.join(PASTA_DADOS, "srag_2025_completo.csv"))

    coluna_uf = "SG_UF_NOT"
    coluna_data = "DT_NOTIFIC"
    coluna_municipio_resi = "ID_MN_RESI"    # município de RESIDÊNCIA do paciente (chave geográfica)
    coluna_municipio_notif = "ID_MUNICIP"   # município de NOTIFICAÇÃO (mantido só de referência)
    coluna_regional = "ID_REGIONA"          # Regional de Saúde de notificação (agrupamento oficial do SUS)

    for nome_var, coluna in [
        ("coluna_uf", coluna_uf), ("coluna_data", coluna_data),
        ("coluna_municipio_resi", coluna_municipio_resi),
        ("coluna_municipio_notif", coluna_municipio_notif),
    ]:
        if coluna not in df.columns:
            print(f"\nA coluna '{coluna}' (usada como {nome_var}) não existe no arquivo.")
            print("Colunas disponíveis:", list(df.columns))
            raise KeyError(f"Ajuste {nome_var} em carregar_srag() com um dos nomes acima.")

    df = df[df[coluna_uf] == UF_ALVO]

    df["data_notificacao"] = pd.to_datetime(df[coluna_data], errors="coerce")
    df["semana"] = df["data_notificacao"].dt.to_period("W")

    # Checagem: ID_MN_RESI precisa vir como NOME de município (texto), igual
    # ID_MUNICIP, pra normalizar_municipio() funcionar. Se vier como código
    # numérico do IBGE, o aviso abaixo aparece — nesse caso normalizar_municipio
    # não resolve sozinho, precisaria de uma tabela código -> nome antes.
    amostra = df[coluna_municipio_resi].dropna().astype(str).head(5).tolist()
    print(f"Amostra de valores em {coluna_municipio_resi}: {amostra}")
    if amostra and all(v.replace(".", "").isdigit() for v in amostra):
        print(f"ATENÇÃO: '{coluna_municipio_resi}' parece conter CÓDIGOS numéricos do IBGE, não "
              f"nomes de município — normalizar_municipio() não vai funcionar direito nesse caso.")

    df["municipio"] = df[coluna_municipio_resi].apply(normalizar_municipio)              # chave = residência
    df["municipio_notificacao"] = df[coluna_municipio_notif].apply(normalizar_municipio)  # mantido pra comparação

    if coluna_regional in df.columns:
        df["regional_saude"] = df[coluna_regional].fillna("Não informado")
    else:
        print(f"Aviso: coluna '{coluna_regional}' não encontrada — regional_saude ficará como 'Não informado'.")
        df["regional_saude"] = "Não informado"

    # ── Decodificação dos campos categóricos (dicionário oficial SIVEP-Gripe) ──
    df["classi_fin_decodificado"] = decodificar_categoria(df["CLASSI_FIN"], MAPA_CLASSI_FIN)
    df["evolucao_decodificada"] = decodificar_categoria(df["EVOLUCAO"], MAPA_EVOLUCAO)
    df["uti_decodificado"] = decodificar_categoria(df["UTI"], MAPA_UTI)
    df["vacina_decodificado"] = decodificar_categoria(df["VACINA"], MAPA_VACINA)
    df["tp_idade_decodificado"] = decodificar_categoria(df["TP_IDADE"], MAPA_TP_IDADE)

    colunas_comorbidade = [
        "CARDIOPATI", "DIABETES", "ASMA", "OBESIDADE",
        "IMUNODEPRE", "RENAL", "PNEUMOPATI", "NEUROLOGIC",
    ]
    for coluna in colunas_comorbidade:
        df[f"{coluna.lower()}_decodificado"] = decodificar_categoria(df[coluna], MAPA_SIM_NAO)

    # ── Decodificação dos campos novos (gravidade, testagem, risco, sintomas) ──
    df["surto_decodificado"] = decodificar_categoria(serie_segura(df, "SURTO_SG"), MAPA_SIM_NAO)
    df["saturacao_decodificada"] = decodificar_categoria(serie_segura(df, "SATURACAO"), MAPA_SIM_NAO)
    df["dispneia_decodificada"] = decodificar_categoria(serie_segura(df, "DISPNEIA"), MAPA_SIM_NAO)
    df["desc_resp_decodificado"] = decodificar_categoria(serie_segura(df, "DESC_RESP"), MAPA_SIM_NAO)
    df["suport_ven_decodificado"] = decodificar_categoria(serie_segura(df, "SUPORT_VEN"), MAPA_SUPORT_VEN)
    df["pcr_resul_decodificado"] = decodificar_categoria(serie_segura(df, "PCR_RESUL"), MAPA_PCR_RESUL)
    df["res_an_decodificado"] = decodificar_categoria(serie_segura(df, "RES_AN"), MAPA_RES_AN)
    df["amostra_decodificada"] = decodificar_categoria(serie_segura(df, "AMOSTRA"), MAPA_SIM_NAO)
    df["fator_risc_decodificado"] = decodificar_categoria(serie_segura(df, "FATOR_RISC"), MAPA_SIM_NAO)
    df["cs_gestant_decodificado"] = decodificar_categoria(serie_segura(df, "CS_GESTANT"), MAPA_CS_GESTANT)
    df["tabag_decodificado"] = decodificar_categoria(serie_segura(df, "TABAG"), MAPA_SIM_NAO)
    df["cs_zona_decodificada"] = decodificar_categoria(serie_segura(df, "CS_ZONA"), MAPA_CS_ZONA)
    df["vacina_cov_decodificada"] = decodificar_categoria(serie_segura(df, "VACINA_COV"), MAPA_SIM_NAO)
    df["febre_decodificada"] = decodificar_categoria(serie_segura(df, "FEBRE"), MAPA_SIM_NAO)
    df["tosse_decodificada"] = decodificar_categoria(serie_segura(df, "TOSSE"), MAPA_SIM_NAO)
    df["an_sars2_decodificado"] = decodificar_categoria(serie_segura(df, "AN_SARS2"), MAPA_SIM_NAO)
    df["pcr_sars2_decodificado"] = decodificar_categoria(serie_segura(df, "PCR_SARS2"), MAPA_SIM_NAO)
    df["an_vsr_decodificado"] = decodificar_categoria(serie_segura(df, "AN_VSR"), MAPA_SIM_NAO)
    df["pcr_vsr_decodificado"] = decodificar_categoria(serie_segura(df, "PCR_VSR"), MAPA_SIM_NAO)

    # ── Indicadores derivados, calculados a partir dos valores decodificados ──
    df["eh_influenza"] = df["classi_fin_decodificado"] == "SRAG por influenza"
    df["eh_covid"] = df["classi_fin_decodificado"] == "SRAG por covid-19"
    df["eh_obito"] = df["evolucao_decodificada"].isin(["Óbito", "Óbito por outras causas"])
    df["foi_uti"] = df["uti_decodificado"] == "Sim"
    df["foi_vacinado"] = df["vacina_decodificado"] == "Sim"

    tem_comorbidade = pd.Series(False, index=df.index)
    for coluna in colunas_comorbidade:
        tem_comorbidade = tem_comorbidade | (df[f"{coluna.lower()}_decodificado"] == "Sim")
    df["tem_comorbidade"] = tem_comorbidade

    # Idade: NU_IDADE_N só representa "anos" quando TP_IDADE decodificado == "Ano"
    idade_em_anos = pd.to_numeric(df["NU_IDADE_N"], errors="coerce").where(
        df["tp_idade_decodificado"] == "Ano"
    )
    df["eh_idoso"] = idade_em_anos >= 60
    # NOVO: idade_anos guardada linha a linha para virar idade_media na agregação
    df["idade_anos"] = idade_em_anos

    # NOVO: CO_REGIONA — código numérico oficial da regional de saúde no
    # SIVEP-Gripe, complementar ao nome já usado em regional_saude
    df["co_regiona_num"] = pd.to_numeric(serie_segura(df, "CO_REGIONA"), errors="coerce")

    # ── Indicadores derivados novos ──
    df["eh_surto"] = df["surto_decodificado"] == "Sim"
    df["eh_saturacao_baixa"] = df["saturacao_decodificada"] == "Sim"
    df["eh_dispneia"] = df["dispneia_decodificada"] == "Sim"
    df["eh_desc_resp"] = df["desc_resp_decodificado"] == "Sim"
    df["eh_vent_invasiva"] = df["suport_ven_decodificado"] == "Sim, invasivo"
    df["eh_pcr_positivo"] = df["pcr_resul_decodificado"] == "Detectável"
    df["eh_antigenico_positivo"] = df["res_an_decodificado"] == "Positivo"
    # SARS2/VSR: "Sim" se QUALQUER um dos dois testes (antígeno OU PCR) deu positivo
    df["eh_sars2"] = (df["an_sars2_decodificado"] == "Sim") | (df["pcr_sars2_decodificado"] == "Sim")
    df["eh_vsr"] = (df["an_vsr_decodificado"] == "Sim") | (df["pcr_vsr_decodificado"] == "Sim")
    df["foi_testado"] = df["amostra_decodificada"] == "Sim"
    df["tem_fator_risco"] = df["fator_risc_decodificado"] == "Sim"
    df["eh_gestante"] = df["cs_gestant_decodificado"].isin(
        ["1º trimestre", "2º trimestre", "3º trimestre", "Idade gestacional ignorada"]
    )
    df["eh_tabagista"] = df["tabag_decodificado"] == "Sim"
    df["eh_zona_rural"] = df["cs_zona_decodificada"] == "Rural"
    df["foi_vacinado_covid"] = df["vacina_cov_decodificada"] == "Sim"
    df["teve_febre"] = df["febre_decodificada"] == "Sim"
    df["teve_tosse"] = df["tosse_decodificada"] == "Sim"

    # Município de internação, pra medir fluxo de pacientes referenciados pra
    # fora do próprio município de residência (indice_fluxo_referencia_hospitalar)
    df["municipio_internacao"] = serie_segura(df, "ID_MN_INTE").apply(
        lambda v: normalizar_municipio(v) if pd.notna(v) else v
    )
    df["foi_referenciado"] = (
        df["municipio_internacao"].notna()
        & (df["municipio_internacao"] != df["municipio"])
    )

    # Datas auxiliares, usadas nos cálculos de atraso/tempo abaixo
    df["dt_sin_pri"] = pd.to_datetime(serie_segura(df, "DT_SIN_PRI"), errors="coerce")
    df["dt_coleta"] = pd.to_datetime(serie_segura(df, "DT_COLETA"), errors="coerce")
    df["dt_interna"] = pd.to_datetime(serie_segura(df, "DT_INTERNA"), errors="coerce")
    df["dt_digita"] = pd.to_datetime(serie_segura(df, "DT_DIGITA"), errors="coerce")
    df["dt_evoluca"] = pd.to_datetime(serie_segura(df, "DT_EVOLUCA"), errors="coerce")
    df["dt_entuti"] = pd.to_datetime(serie_segura(df, "DT_ENTUTI"), errors="coerce")
    df["dt_saiduti"] = pd.to_datetime(serie_segura(df, "DT_SAIDUTI"), errors="coerce")

    df["dias_ate_notificacao"] = (df["data_notificacao"] - df["dt_sin_pri"]).dt.days
    df["dias_ate_coleta"] = (df["dt_coleta"] - df["dt_sin_pri"]).dt.days
    df["dias_ate_internacao"] = (df["dt_interna"] - df["dt_sin_pri"]).dt.days
    df["dias_uti"] = (df["dt_saiduti"] - df["dt_entuti"]).dt.days
    df["dias_ate_digitacao"] = (df["dt_digita"] - df["data_notificacao"]).dt.days
    df["dias_ate_desfecho"] = (df["dt_evoluca"] - df["dt_interna"]).dt.days

    def primeiro_nao_nulo(s):
        s = s.dropna()
        return s.iloc[0] if len(s) else np.nan

    agregados = df.groupby(["municipio", "semana"]).agg(
        regional_saude=("regional_saude", "first"),
        idade_media=("idade_anos", "mean"),          # NOVO
        CO_REGIONA=("co_regiona_num", primeiro_nao_nulo),  # NOVO
        casos_srag=("municipio", "size"),
        casos_influenza=("eh_influenza", "sum"),
        casos_covid=("eh_covid", "sum"),
        obitos=("eh_obito", "sum"),
        casos_uti=("foi_uti", "sum"),
        casos_com_comorbidade=("tem_comorbidade", "sum"),
        casos_vacinados=("foi_vacinado", "sum"),
        casos_idosos=("eh_idoso", "sum"),
        casos_surto=("eh_surto", "sum"),
        casos_saturacao_baixa=("eh_saturacao_baixa", "sum"),
        casos_dispneia=("eh_dispneia", "sum"),
        casos_desc_resp=("eh_desc_resp", "sum"),
        casos_vent_invasiva=("eh_vent_invasiva", "sum"),
        tempo_medio_uti_dias=("dias_uti", "mean"),
        casos_pcr_positivo=("eh_pcr_positivo", "sum"),
        casos_antigenico_positivo=("eh_antigenico_positivo", "sum"),
        casos_sars2=("eh_sars2", "sum"),
        casos_vsr=("eh_vsr", "sum"),
        casos_testados=("foi_testado", "sum"),
        atraso_notificacao_dias=("dias_ate_notificacao", "mean"),
        atraso_coleta_dias=("dias_ate_coleta", "mean"),
        atraso_internacao_dias=("dias_ate_internacao", "mean"),
        casos_fator_risco=("tem_fator_risco", "sum"),
        casos_gestantes=("eh_gestante", "sum"),
        casos_tabagistas=("eh_tabagista", "sum"),
        casos_zona_rural=("eh_zona_rural", "sum"),
        casos_vacinados_covid=("foi_vacinado_covid", "sum"),
        casos_febre=("teve_febre", "sum"),
        casos_tosse=("teve_tosse", "sum"),
        atraso_digitacao_dias=("dias_ate_digitacao", "mean"),
        tempo_ate_desfecho_dias=("dias_ate_desfecho", "mean"),
        casos_referenciados=("foi_referenciado", "sum"),
    ).reset_index()

    agregados["taxa_obito"] = agregados["obitos"] / agregados["casos_srag"]
    agregados["taxa_uti"] = agregados["casos_uti"] / agregados["casos_srag"]
    agregados["taxa_comorbidade"] = agregados["casos_com_comorbidade"] / agregados["casos_srag"]
    agregados["taxa_vacinados"] = agregados["casos_vacinados"] / agregados["casos_srag"]
    agregados["taxa_idosos"] = agregados["casos_idosos"] / agregados["casos_srag"]

    agregados["taxa_cadeia_surto"] = agregados["casos_surto"] / agregados["casos_srag"]
    agregados["taxa_saturacao_baixa"] = agregados["casos_saturacao_baixa"] / agregados["casos_srag"]
    agregados["taxa_dispneia"] = agregados["casos_dispneia"] / agregados["casos_srag"]
    agregados["taxa_desc_respiratorio"] = agregados["casos_desc_resp"] / agregados["casos_srag"]
    agregados["taxa_ventilacao_invasiva"] = agregados["casos_vent_invasiva"] / agregados["casos_srag"]
    agregados["taxa_positividade_pcr"] = (
        agregados["casos_pcr_positivo"] / agregados["casos_testados"].replace(0, pd.NA)
    )
    agregados["taxa_positividade_antigenico"] = (
        agregados["casos_antigenico_positivo"] / agregados["casos_testados"].replace(0, pd.NA)
    )
    agregados["taxa_sars2"] = agregados["casos_sars2"] / agregados["casos_srag"]
    agregados["taxa_vsr"] = agregados["casos_vsr"] / agregados["casos_srag"]
    agregados["taxa_testagem"] = agregados["casos_testados"] / agregados["casos_srag"]
    agregados["taxa_fator_risco"] = agregados["casos_fator_risco"] / agregados["casos_srag"]
    agregados["taxa_gestantes"] = agregados["casos_gestantes"] / agregados["casos_srag"]
    agregados["taxa_tabagismo"] = agregados["casos_tabagistas"] / agregados["casos_srag"]
    agregados["taxa_zona_rural"] = agregados["casos_zona_rural"] / agregados["casos_srag"]
    agregados["taxa_vacinacao_covid"] = agregados["casos_vacinados_covid"] / agregados["casos_srag"]
    agregados["taxa_febre"] = agregados["casos_febre"] / agregados["casos_srag"]
    agregados["taxa_tosse"] = agregados["casos_tosse"] / agregados["casos_srag"]
    agregados["indice_fluxo_referencia_hospitalar"] = agregados["casos_referenciados"] / agregados["casos_srag"]

    print("SRAG: OK")
    return agregados


## Etapa 3 — Leitos (filtrada para SP)

Sem mudança na granularidade temporal (leitos é um dado por município, não por período). Mantém
leitos por especialidade de UTI, proporção SUS e contagem de hospitais distintos.

In [7]:
def carregar_leitos():
    print("Carregando Leitos...")
    df = ler_csv_seguro(os.path.join(PASTA_DADOS, "Leitos_2025.csv"))

    coluna_uf = "UF"
    coluna_municipio = "MUNICIPIO"
    coluna_leitos = "LEITOS_SUS"
    coluna_leitos_existentes = "LEITOS_EXISTENTES"
    coluna_cnes = "CNES"

    for nome_var, coluna in [
        ("coluna_uf", coluna_uf), ("coluna_municipio", coluna_municipio), ("coluna_leitos", coluna_leitos),
    ]:
        if coluna not in df.columns:
            print(f"\nA coluna '{coluna}' (usada como {nome_var}) não existe no arquivo.")
            print("Colunas disponíveis:", list(df.columns))
            raise KeyError(f"Ajuste {nome_var} em carregar_leitos() com um dos nomes acima.")

    df = df[df[coluna_uf] == UF_ALVO]
    df["municipio"] = df[coluna_municipio].apply(normalizar_municipio)

    df[coluna_leitos] = pd.to_numeric(df[coluna_leitos], errors="coerce")

    colunas_uti_especialidade = {
        "UTI_TOTAL_SUS": "leitos_uti_sus",
        "UTI_ADULTO_SUS": "leitos_uti_adulto_sus",
        "UTI_PEDIATRICO_SUS": "leitos_uti_pediatrico_sus",
        "UTI_NEONATAL_SUS": "leitos_uti_neonatal_sus",
    }
    agregacoes = {coluna_leitos: "sum"}
    for coluna_original in colunas_uti_especialidade:
        if coluna_original in df.columns:
            df[coluna_original] = pd.to_numeric(df[coluna_original], errors="coerce")
            agregacoes[coluna_original] = "sum"
        else:
            print(f"Aviso: coluna '{coluna_original}' não encontrada — pulando essa especialidade.")

    if coluna_leitos_existentes in df.columns:
        df[coluna_leitos_existentes] = pd.to_numeric(df[coluna_leitos_existentes], errors="coerce")
        agregacoes[coluna_leitos_existentes] = "sum"
    else:
        print(f"Aviso: coluna '{coluna_leitos_existentes}' não encontrada — não será possível calcular proporcao_leitos_sus.")

    ocupacao = df.groupby("municipio").agg(agregacoes).reset_index()
    ocupacao = ocupacao.rename(columns=colunas_uti_especialidade)

    if coluna_leitos_existentes in ocupacao.columns:
        ocupacao["proporcao_leitos_sus"] = ocupacao[coluna_leitos] / ocupacao[coluna_leitos_existentes]

    if coluna_cnes in df.columns:
        hospitais_sus = (
            df[df[coluna_leitos] > 0]
            .groupby("municipio")[coluna_cnes]
            .nunique()
            .reset_index(name="n_hospitais_sus")
        )
        ocupacao = ocupacao.merge(hospitais_sus, on="municipio", how="left")
        ocupacao["n_hospitais_sus"] = ocupacao["n_hospitais_sus"].fillna(0)
    else:
        print(f"Aviso: coluna '{coluna_cnes}' não encontrada — não será possível calcular n_hospitais_sus.")

    print("Leitos: OK")
    return ocupacao


## Etapa 4 — População (IBGE)

O JSON vem no formato aninhado da API do IBGE (SIDRA).

In [8]:
def carregar_populacao():
    caminho = os.path.join(PASTA_DADOS, "populacao_municipios_2025 (1).json")
    print("Carregando População...")
    with open(caminho, encoding="utf-8") as f:
        dados = json.load(f)

    series = dados[0]["resultados"][0]["series"]

    registros = []
    for item in series:
        nome_completo = item["localidade"]["nome"]
        populacao = item["serie"].get("2025")
        if " - " in nome_completo:
            municipio, uf = nome_completo.rsplit(" - ", 1)
        else:
            municipio, uf = nome_completo, None
        registros.append({
            "municipio": normalizar_municipio(municipio),
            "uf": uf,
            "populacao": int(populacao) if populacao is not None else None,
        })

    populacao_df = pd.DataFrame(registros)
    populacao_df = populacao_df[populacao_df["uf"] == UF_ALVO]
    print("População: OK")
    return populacao_df[["municipio", "populacao"]]


## Classificação regional (Norte / Sul / Leste / Oeste / Centro)

Reaproveita a geocodificação já feita para a imputação de temperatura — cada município é
classificado com base na posição da sua coordenada em relação ao **centro geográfico do estado**
(mediana das coordenadas de todos os municípios do dataset), não em relação à capital.

**Como funciona:** calcula o centro (mediana de latitude/longitude) e uma faixa central em torno
dele. Município dentro dessa faixa nos dois eixos vira `"Centro"`. Fora da faixa, compara se o
desvio é maior em latitude (Norte/Sul) ou em longitude (Leste/Oeste) e classifica pelo eixo
dominante. Município sem coordenada encontrada vira `"Não informado"`.

In [9]:
def geocodificar_municipios(lista_municipios, pausa_segundos=1.0):
    """Descobre a latitude/longitude de cada município via OpenStreetMap
    (Nominatim). Respeita 1 requisição por segundo (limite do serviço gratuito)."""
    from geopy.geocoders import Nominatim
    geolocator = Nominatim(user_agent="vigiar_etl_fiap")
    coordenadas = {}
    for i, municipio in enumerate(lista_municipios, start=1):
        try:
            local = geolocator.geocode(f"{municipio}, São Paulo, Brasil", timeout=10)
            coordenadas[municipio] = {"lat": local.latitude, "lon": local.longitude} if local else {"lat": None, "lon": None}
        except Exception:
            coordenadas[municipio] = {"lat": None, "lon": None}
        time.sleep(pausa_segundos)
        if i % 50 == 0:
            print(f"  {i}/{len(lista_municipios)} municípios geocodificados")
    return coordenadas


def classificar_regiao(municipios_coords):
    """Classifica cada município em Norte/Sul/Leste/Oeste/Centro, com base na
    posição em relação à mediana das coordenadas de todos os municípios (o
    'centro geográfico' do conjunto de dados)."""
    lats = [c["lat"] for c in municipios_coords.values() if c["lat"] is not None]
    lons = [c["lon"] for c in municipios_coords.values() if c["lon"] is not None]
    if not lats or not lons:
        return {m: "Não informado" for m in municipios_coords}

    lat_central, lon_central = np.median(lats), np.median(lons)
    margem_lat = (max(lats) - min(lats)) / 6 or 0.01
    margem_lon = (max(lons) - min(lons)) / 6 or 0.01

    regioes = {}
    for municipio, coord in municipios_coords.items():
        if coord["lat"] is None or coord["lon"] is None:
            regioes[municipio] = "Não informado"
            continue
        delta_lat = coord["lat"] - lat_central
        delta_lon = coord["lon"] - lon_central
        if abs(delta_lat) < margem_lat and abs(delta_lon) < margem_lon:
            regioes[municipio] = "Centro"
        elif abs(delta_lat) / margem_lat >= abs(delta_lon) / margem_lon:
            regioes[municipio] = "Norte" if delta_lat > 0 else "Sul"
        else:
            regioes[municipio] = "Leste" if delta_lon > 0 else "Oeste"
    return regioes


## Grade completa de município x semana + união final

Antes de unir, criamos a **grade completa** de município × semana — todo município tem uma linha em
toda semana do período, mesmo sem notificação de SRAG (nesse caso, contagens viram 0, em vez da
linha simplesmente não existir). População, leitos e temperatura continuam podendo ficar vazios
quando não há dado real disponível.

In [10]:
def completar_grade_e_unir(srag, populacao, leitos, temperatura):
    # Filtro adicionado após investigação: como "municipio" agora vem do município de
    # RESIDÊNCIA (ID_MN_RESI), pacientes que moram em outro estado mas foram atendidos em
    # SP (filtro de UF é sobre a notificação, não a residência) entravam na base. Isso não
    # só inflava o dataset com municípios fora de escopo — contaminava a geocodificação e
    # a classificação regional: max(lat)/min(lat) passava a medir a extensão do BRASIL
    # inteiro em vez de só SP, e a margem de "Centro" ficava tão grande que 91% dos
    # municípios paulistas caíam em "Centro" e Norte/Sul ficavam vazios. Filtrando aqui,
    # antes de montar todos_municipios, o problema é resolvido na raiz — tanto pro dataset
    # final quanto pra geocodificação/classificação regional que usa essa mesma lista.
    municipios_sp = set(populacao["municipio"])
    n_municipios_antes = srag["municipio"].nunique()
    n_linhas_antes = len(srag)
    srag = srag[srag["municipio"].isin(municipios_sp)].copy()
    n_municipios_removidos = n_municipios_antes - srag["municipio"].nunique()
    n_linhas_removidas = n_linhas_antes - len(srag)
    if n_municipios_removidos > 0:
        print(f"SRAG: removidos {n_municipios_removidos} municípios ({n_linhas_removidas} linhas) "
              f"sem correspondência na tabela de população — residência fora de SP ou não informada.")

    todos_municipios = pd.concat([
        srag["municipio"], populacao["municipio"], leitos["municipio"]
    ]).unique()
    semanas = pd.period_range(srag["semana"].min(), srag["semana"].max(), freq="W")

    grade = pd.MultiIndex.from_product(
        [todos_municipios, semanas], names=["municipio", "semana"]
    ).to_frame(index=False)

    dataset = grade.merge(srag, on=["municipio", "semana"], how="left")
    dataset = dataset.merge(populacao, on="municipio", how="left")
    dataset = dataset.merge(leitos, on="municipio", how="left")
    dataset = dataset.merge(temperatura, on=["municipio", "semana"], how="left")

    colunas_contagem = [
        "casos_srag", "casos_influenza", "casos_covid", "obitos",
        "casos_uti", "casos_com_comorbidade", "casos_vacinados", "casos_idosos",
        "taxa_obito", "taxa_uti", "taxa_comorbidade", "taxa_vacinados", "taxa_idosos",
        # novas variáveis: contagens e taxas (0 é o valor correto quando não há
        # casos na semana). As colunas de dias (tempo_medio_uti_dias,
        # atraso_*_dias, tempo_ate_desfecho_dias) ficam de fora de propósito —
        # continuam NaN quando não há caso, porque "0 dias de atraso" seria
        # uma informação errada, não "não houve caso".
        "casos_surto", "casos_saturacao_baixa", "casos_dispneia", "casos_desc_resp",
        "casos_vent_invasiva", "casos_pcr_positivo", "casos_antigenico_positivo",
        "casos_sars2", "casos_vsr", "casos_testados", "casos_fator_risco",
        "casos_gestantes", "casos_tabagistas", "casos_zona_rural",
        "casos_vacinados_covid", "casos_febre", "casos_tosse", "casos_referenciados",
        "taxa_cadeia_surto", "taxa_saturacao_baixa", "taxa_dispneia",
        "taxa_desc_respiratorio", "taxa_ventilacao_invasiva",
        "taxa_positividade_pcr", "taxa_positividade_antigenico", "taxa_sars2",
        "taxa_vsr", "taxa_testagem", "taxa_fator_risco", "taxa_gestantes",
        "taxa_tabagismo", "taxa_zona_rural", "taxa_vacinacao_covid",
        "taxa_febre", "taxa_tosse", "indice_fluxo_referencia_hospitalar",
    ]
    for coluna in colunas_contagem:
        if coluna in dataset.columns:
            dataset[coluna] = dataset[coluna].fillna(0)

    return dataset, todos_municipios


## Geração do dataset final

Une tudo, geocodifica os municípios (uma única vez, reaproveitada para região e temperatura),
imputa temperatura ausente por proximidade, calcula colunas derivadas e formata as datas.

In [11]:
def preencher_temperatura_por_proximidade(dataset, temperatura, estacoes_coords, municipios_coords):
    """Para linhas sem temp_media, usa a temperatura da estação mais próxima
    do município (calculada por coordenada), marcando a linha em
    'temp_imputada'."""
    dataset = dataset.copy()
    dataset["temp_imputada"] = dataset["temp_media"].isna()

    def estacao_mais_proxima(lat, lon):
        melhor, menor_dist = None, float("inf")
        for nome_estacao, info in estacoes_coords.items():
            if info["lat"] is None or info["lon"] is None:
                continue
            R = 6371
            dlat, dlon = radians(info["lat"] - lat), radians(info["lon"] - lon)
            a = sin(dlat / 2) ** 2 + cos(radians(lat)) * cos(radians(info["lat"])) * sin(dlon / 2) ** 2
            dist = R * 2 * atan2(sqrt(a), sqrt(1 - a))
            if dist < menor_dist:
                menor_dist, melhor = dist, nome_estacao
        return melhor

    mapa_estacao_mais_proxima = {}
    for municipio in dataset.loc[dataset["temp_imputada"], "municipio"].unique():
        coord = municipios_coords.get(municipio, {"lat": None, "lon": None})
        if coord["lat"] is None:
            mapa_estacao_mais_proxima[municipio] = None
        else:
            mapa_estacao_mais_proxima[municipio] = estacao_mais_proxima(coord["lat"], coord["lon"])

    def buscar_temp_proxima(linha):
        if not linha["temp_imputada"]:
            return linha["temp_media"]
        estacao_proxima = mapa_estacao_mais_proxima.get(linha["municipio"])
        if estacao_proxima is None:
            return None
        valores = temperatura.loc[
            (temperatura["municipio"] == estacao_proxima) & (temperatura["semana"] == linha["semana"]),
            "temp_media"
        ]
        return valores.iloc[0] if len(valores) else None

    dataset["temp_media"] = dataset.apply(buscar_temp_proxima, axis=1)
    n_imputadas = dataset.loc[dataset["temp_imputada"], "temp_media"].notna().sum()
    print(f"{n_imputadas} de {dataset['temp_imputada'].sum()} linhas preenchidas por proximidade.")
    return dataset


def gerar_dataset_final():
    print("Carregando e tratando cada fonte (recorte: São Paulo)...")
    temperatura, estacoes_coords = carregar_temperatura_sp()
    srag = carregar_srag()
    leitos = carregar_leitos()
    populacao = carregar_populacao()

    print("Completando a grade de município x semana (semanas sem SRAG viram 0)...")
    dataset, todos_municipios = completar_grade_e_unir(srag, populacao, leitos, temperatura)

    print(f"Geocodificando {len(todos_municipios)} municípios (usado para região + temperatura)...")
    municipios_coords = geocodificar_municipios(todos_municipios)

    print("Classificando região de cada município...")
    mapa_regiao = classificar_regiao(municipios_coords)
    dataset["regiao"] = dataset["municipio"].map(mapa_regiao).fillna("Não informado")

    print("Preenchendo temperaturas ausentes por proximidade geográfica...")
    dataset = preencher_temperatura_por_proximidade(dataset, temperatura, estacoes_coords, municipios_coords)

    print("Calculando colunas derivadas (sazonalidade, capacidade, tendência)...")
    dataset = dataset.sort_values(["municipio", "semana"]).reset_index(drop=True)

    dataset["casos_por_10k"] = dataset["casos_srag"] / dataset["populacao"] * 10_000
    dataset["leitos_por_1000_hab"] = dataset["LEITOS_SUS"] / dataset["populacao"] * 1000

    dataset["data_inicio_semana"] = dataset["semana"].dt.start_time
    dataset["ano"] = dataset["data_inicio_semana"].dt.year
    dataset["semana_numero"] = dataset["data_inicio_semana"].dt.isocalendar().week
    dataset["data_formatada"] = dataset["data_inicio_semana"].apply(formatar_data_br)

    dataset["casos_srag_semana_anterior"] = dataset.groupby("municipio")["casos_srag"].shift(1)
    dataset["media_movel_3_semanas_casos"] = dataset.groupby("municipio")["casos_srag"].transform(
        lambda serie: serie.shift(1).rolling(window=3, min_periods=1).mean()
    )
    dataset["variacao_temp"] = dataset.groupby("municipio")["temp_media"].diff()
    dataset["variacao_casos_pct"] = (
        dataset.groupby("municipio")["casos_srag"].pct_change()
        .replace([float("inf"), float("-inf")], None)
    )

    dataset = dataset.drop(columns=["semana", "data_inicio_semana"])

    colunas_finais = [
        "municipio", "regiao", "regional_saude", "ano", "semana_numero", "data_formatada",
        "casos_srag", "casos_influenza", "casos_covid", "obitos", "casos_uti",
        "casos_com_comorbidade", "casos_vacinados", "casos_idosos",
        "taxa_obito", "taxa_uti", "taxa_comorbidade", "taxa_vacinados", "taxa_idosos",
        "populacao", "LEITOS_SUS", "leitos_uti_sus", "leitos_uti_adulto_sus",
        "leitos_uti_pediatrico_sus", "leitos_uti_neonatal_sus", "LEITOS_EXISTENTES",
        "proporcao_leitos_sus", "n_hospitais_sus", "leitos_por_1000_hab",
        "temp_media", "temp_imputada", "casos_por_10k",
        "casos_srag_semana_anterior", "media_movel_3_semanas_casos",
        "variacao_temp", "variacao_casos_pct",
        # 24 novas variáveis (tabela de features anexada na conversa)
        "taxa_cadeia_surto", "taxa_saturacao_baixa", "taxa_dispneia",
        "taxa_desc_respiratorio", "taxa_ventilacao_invasiva", "tempo_medio_uti_dias",
        "taxa_positividade_pcr", "taxa_positividade_antigenico", "taxa_sars2", "taxa_vsr",
        "taxa_testagem", "atraso_notificacao_dias", "atraso_coleta_dias",
        "indice_fluxo_referencia_hospitalar", "atraso_internacao_dias",
        "taxa_fator_risco", "taxa_gestantes", "taxa_tabagismo", "taxa_zona_rural",
        "taxa_vacinacao_covid", "taxa_febre", "taxa_tosse",
        "atraso_digitacao_dias", "tempo_ate_desfecho_dias",
        # 2 colunas novas desta rodada:
        "idade_media", "CO_REGIONA",
    ]
    colunas_finais = [c for c in colunas_finais if c in dataset.columns]
    dataset = dataset[colunas_finais]

    dataset.to_csv(SAIDA, index=False)
    print(f"\nDataset final salvo em: {SAIDA}")
    print(f"Municípios únicos: {dataset['municipio'].nunique()}")
    print(f"Total de linhas: {len(dataset)}")
    return dataset


dataset_final = gerar_dataset_final()
display(dataset_final.head())


Carregando e tratando cada fonte (recorte: São Paulo)...
40 estações de SP encontradas
Temperatura: OK
Carregando SRAG...
Aviso: srag_2025_completo.csv teve linha(s) malformada(s) — pulando e continuando...


/tmp/ipykernel_906/1895263606.py:24: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["semana"] = df["data_notificacao"].dt.to_period("W")


Amostra de valores em ID_MN_RESI: ['SAO PAULO', 'SAO PAULO', 'SAO PAULO', 'GUARULHOS', 'SAO VICENTE']
SRAG: OK
Carregando Leitos...
Leitos: OK
Carregando População...
População: OK
Completando a grade de município x semana (semanas sem SRAG viram 0)...
SRAG: removidos 123 municípios (203 linhas) sem correspondência na tabela de população — residência fora de SP ou não informada.


/tmp/ipykernel_906/1361943402.py:58: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset[coluna] = dataset[coluna].fillna(0)


Geocodificando 645 municípios (usado para região + temperatura)...
  50/645 municípios geocodificados
  100/645 municípios geocodificados
  150/645 municípios geocodificados
  200/645 municípios geocodificados
  250/645 municípios geocodificados
  300/645 municípios geocodificados
  350/645 municípios geocodificados
  400/645 municípios geocodificados
  450/645 municípios geocodificados
  500/645 municípios geocodificados
  550/645 municípios geocodificados
  600/645 municípios geocodificados
Classificando região de cada município...
Preenchendo temperaturas ausentes por proximidade geográfica...
29521 de 52300 linhas preenchidas por proximidade.
Calculando colunas derivadas (sazonalidade, capacidade, tendência)...

Dataset final salvo em: /content/dataset_vigiar_sp.csv
Municípios únicos: 645
Total de linhas: 54180


,municipio,regiao,regional_saude,ano,semana_numero,data_formatada,casos_srag,casos_influenza,casos_covid,obitos,...,taxa_gestantes,taxa_tabagismo,taxa_zona_rural,taxa_vacinacao_covid,taxa_febre,taxa_tosse,atraso_digitacao_dias,tempo_ate_desfecho_dias,idade_media,CO_REGIONA
0,ADAMANTINA,Oeste,NaN,2024,52,23/12/2024,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
1,ADAMANTINA,Oeste,NaN,2024,1,30/12/2024,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
2,ADAMANTINA,Oeste,NaN,2025,2,06/01/2025,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
3,ADAMANTINA,Oeste,NaN,2025,3,13/01/2025,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
4,ADAMANTINA,Oeste,NaN,2025,4,20/01/2025,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN


## Download do resultado

In [12]:
from google.colab import files

files.download(SAIDA)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>